In [6]:
import pandas as pd
import numpy as np

In [ ]:
movies=pd.read_csv("tmdb_5000_movies.csv")

In [ ]:
credits=pd.read_csv("tmdb_5000_credits.csv")

In [ ]:
movies.head(2)

In [ ]:
credits.head(2)

### **Merging both tables**

In [ ]:
movies=movies.merge(credits,on="title")

In [ ]:
movies.head(2)

### **Features we need.**


* id
* title
* overview
* cast
* crew
* keywords
* genre
* production_companies


In [ ]:
movies=movies[['id','title','overview','cast','crew','keywords','genres','production_companies']]

In [ ]:
movies.isnull().sum()

In [ ]:
movies=movies.dropna()

In [ ]:
movies['crew'].iloc[0]

In [ ]:
#There is string in All Text based columns so will converting it into list.
import ast
def convert(text):
  L=[]
  for i in ast.literal_eval(text):
    L.append(i['name'])
  return L

In [ ]:
movies['genres']=movies['genres'].apply(convert)

In [ ]:
movies.head(2)

In [ ]:
movies['keywords']=movies['keywords'].apply(convert)
movies['production_companies']=movies['production_companies'].apply(convert)


In [ ]:
#Cast is too big so we will just choose top 3 cast.
import ast
def top3(text):
  L=[]
  counter=0
  for i in ast.literal_eval(text):
    if counter<3:
      L.append(i['name'])
    counter+=1
  return L

In [ ]:
movies['cast']=movies['cast'].apply(top3)

In [ ]:
movies.head(2)

In [ ]:
import ast
def director(text):
  L=[]
  for i in ast.literal_eval(text):
    if i['job']=='Director':
      L.append(i['name'])
  return L

In [ ]:
movies["crew"]=movies["crew"].apply(director)

In [ ]:
movies.head(2)

In [ ]:
def collapse(words):
  L=[]
  for i in words:
    L.append(i.replace(" ",""))
  return L

In [ ]:
for i in ['cast','crew','keywords','production_companies','genres']:
  movies[i]=movies[i].apply(collapse)


In [ ]:
movies.head(2)

In [ ]:
movies['overview']=movies['overview'].apply(lambda x : x.split())

In [ ]:
movies["tags"]=movies["cast"]+movies["crew"]+movies["keywords"]+movies["genres"]+movies["production_companies"]+movies["overview"]

In [ ]:
new=movies[["id","title","tags"]]

In [ ]:
new.head()

In [ ]:
new['tags']=new['tags'].apply(lambda x: " ".join(x))

In [ ]:
new['tags'].iloc[0]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words="english")

In [ ]:
vector=cv.fit_transform(new['tags']).toarray()

In [ ]:
vector.shape

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity=cosine_similarity(vector)

In [ ]:
similarity[0:3]

In [ ]:
def recommend(movie):
  index=new[new['title']==movie].index[0]
  distances=sorted(enumerate(similarity[index]), reverse=True, key=lambda x:x[1])
  for i in distances[1:6]:
    print(new.iloc[i[0]].title)

In [ ]:
recommend("Titanic")

In [ ]:
pip install pickle

In [ ]:
import pickle
pickle.dump(new.to_dict(), open("movie_dict.pkl",'wb'))
pickle.dump(similarity, open("similarity.pkl",'wb'))